# Nonreciprocal interactions

Add an asymmetric coupling to the chemical potential,

$$\mu_i = \frac{\delta F}{\delta c_i} + \sum_j B_{ij}\,\phi(c_j), \qquad B_{ij}\neq B_{ji}.$$

`chi` remains the symmetric equilibrium interaction matrix. `B` drives non-variational dynamics. The examples below compare two-species predator-prey-like motion, phase separation with antisymmetric forcing, random multispecies dynamics, spectra, coupling-strength sweeps, and activation choices.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from jax_phase_separation.utils import (
    generate_chi_matrix, generate_B_matrix, generate_initial_conditions,
    build_params, plot_volume_fractions,
)
from jax_phase_separation.solver import simulate, simulate_with_snapshots
from jax_phase_separation.free_energy import compute_jacobian

print('JAX version:', jax.__version__, '  devices:', jax.devices())


In [ ]:
# XLA warmup on tiny grid so the subsequent timing is representative.
_chi = generate_chi_matrix(4, 0.0, 1.0, jax.random.PRNGKey(0))
_c0 = generate_initial_conditions(4, 16, key=jax.random.PRNGKey(0))
_p = build_params(_chi, 4, lmbda=0.01, dt=5e-6, B=jnp.zeros((4, 4)))
_ = simulate(_c0, _p, 16, 10).block_until_ready()
print('warmup done')


## 1. Two-species predator-prey

Use `B = [[0, +b], [-b, 0]]` with weak symmetric `chi`, so patterns come from the nonreciprocal drive rather than an equilibrium spinodal. The setup uses `omit_solvent_flux=True`, a mass-conserving sinusoidal bias, weak diffusion, and an `A` value set by `chi` rather than `B`.

In [ ]:
N_COM = 2
N_GRID = 64
LMBDA = 0.02
DT = 5e-6
BETA = 0.6
# Weak diffusion: D_i ~ 1/r_i.
R_MU_PP = 500.0
N_STEPS_PP = 120_000
SAVE_EVERY_PP = 1_000

key = jax.random.PRNGKey(1)
k_chi, k_ic = jax.random.split(key)

# Weak symmetric chi, below the spinodal threshold.
chi_2 = 1.5 * jnp.array([[0.0, 1.0], [1.0, 0.0]])

# Strong antisymmetric B.
b = 50.0
B_pp = jnp.array([[0.0, +b], [-b, 0.0]])

c0_2 = generate_initial_conditions(N_COM, N_GRID, beta=BETA, noise_strength=0.12, key=k_ic)
# Mass-conserving x-bias to break isotropy.
x1d = jnp.linspace(0.0, 1.0, N_GRID, endpoint=False)
strip = 0.05 * jnp.sin(2.0 * jnp.pi * x1d)
strip2d = strip[jnp.newaxis, :]
c0_2 = c0_2.at[0].add(strip2d).at[1].add(-strip2d)

# Set A from |chi| only.
A_stab = float(jnp.abs(chi_2).max()) * LMBDA
params_2 = build_params(
    chi_2, N_COM, beta=BETA, lmbda=LMBDA, dt=DT, B=B_pp,
    omit_solvent_flux=True,
    A=A_stab,
    r_mu=R_MU_PP,
)

c_final, snaps = simulate_with_snapshots(
    c0_2, params_2, N_GRID, n_steps=N_STEPS_PP, save_every=SAVE_EVERY_PP,
    activation='identity',
    include_initial=True,
)
print('snapshots shape:', snaps.shape)
print('species 0 at t=0: min, max =', float(snaps[0, 0].min()), float(snaps[0, 0].max()))
print('species 0 mid-run spatial std (frame 5) =', float(snaps[5, 0].std()))


In [ ]:
n = snaps.shape[0]
t_labels = np.empty(n)
t_labels[0] = 0.0
t_labels[1:] = np.arange(1, n) * SAVE_EVERY_PP * DT
k = min(8, n)
idx = np.unique(np.rint(np.linspace(0, n - 1, k)).astype(int))
n_cols = len(idx)

fig, axes = plt.subplots(2, n_cols, figsize=(2.2 * n_cols, 4.5), squeeze=False)
vmins = [float(snaps[:, sp].min()) for sp in range(2)]
vmaxs = [float(snaps[:, sp].max()) for sp in range(2)]
for sp in range(2):
    if vmaxs[sp] - vmins[sp] < 1e-9:
        vmins[sp] -= 0.02
        vmaxs[sp] += 0.02
for col, i in enumerate(idx):
    for sp in range(2):
        ax = axes[sp, col]
        ax.imshow(
            snaps[i, sp], origin='lower',
            cmap=['Blues', 'Reds'][sp], vmin=vmins[sp], vmax=vmaxs[sp],
        )
        ax.set_xticks([]); ax.set_yticks([])
        if sp == 0:
            ax.set_title(f't = {t_labels[i]:.3f}')
        if col == 0:
            ax.set_ylabel(f'species {sp}', rotation=0, labelpad=30)
fig.suptitle(
    'Predator (top) chases prey (bottom)  —  antisymmetric B; '
    'A from χ only; true IC at t = 0'
)
fig.tight_layout()
plt.show()


In [ ]:
from matplotlib.animation import FuncAnimation


def movie_two_species(snaps, path, t_labels, fps=10, dpi=110):
    """Side-by-side species 0 / 1 (predator–prey panel)."""
    snaps = np.asarray(snaps)
    n = snaps.shape[0]
    vmins = [float(snaps[:, sp].min()) for sp in range(2)]
    vmaxs = [float(snaps[:, sp].max()) for sp in range(2)]
    for sp in range(2):
        if vmaxs[sp] - vmins[sp] < 1e-9:
            vmins[sp] -= 0.02
            vmaxs[sp] += 0.02

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8, 3.8))
    im0 = ax0.imshow(snaps[0, 0], origin='lower', cmap='Blues', vmin=vmins[0], vmax=vmaxs[0])
    im1 = ax1.imshow(snaps[0, 1], origin='lower', cmap='Reds', vmin=vmins[1], vmax=vmaxs[1])
    for ax in (ax0, ax1):
        ax.set_xticks([])
        ax.set_yticks([])
    ax0.set_ylabel('species 0 (predator)')
    ax1.set_ylabel('species 1 (prey)')
    fig.suptitle('')

    def update(i):
        im0.set_data(snaps[i, 0])
        im1.set_data(snaps[i, 1])
        fig.suptitle(f'2-species nonreciprocal   t = {t_labels[i]:.4f}')
        return im0, im1

    anim = FuncAnimation(fig, update, frames=n, interval=1000 / fps, blit=False)
    anim.save(path, writer='ffmpeg', fps=fps, dpi=dpi)
    plt.close(fig)
    print('saved', path)


def movie_species0(snaps, path, t_labels, suptitle, fps=10, dpi=100, vmin=None, vmax=None):
    """Scalar heatmap movie for species 0."""
    snaps = np.asarray(snaps)
    field = snaps[:, 0]
    if vmin is None:
        vmin = float(field.min())
    if vmax is None:
        vmax = float(field.max())
    if vmax - vmin < 1e-9:
        vmin -= 0.02
        vmax += 0.02

    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(field[0], origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax)
    ax.set_xticks([])
    ax.set_yticks([])

    def update(i):
        im.set_data(field[i])
        ax.set_title(f'{suptitle}   t = {t_labels[i]:.4f}')
        return (im,)

    anim = FuncAnimation(fig, update, frames=snaps.shape[0], interval=1000 / fps, blit=False)
    anim.save(path, writer='ffmpeg', fps=fps, dpi=dpi)
    plt.close(fig)
    print('saved', path)


def movie_sigma_grid(snaps_list, sigmas, path, t_labels, fps=8, dpi=100, vmin=0.0, vmax=0.3):
    """2×3 grid: species 0 for each σ_b value, same time index in all panels."""
    snaps_list = [np.asarray(s) for s in snaps_list]
    n_frames = min(s.shape[0] for s in snaps_list)
    fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
    ims = []
    for ax, snaps, s in zip(axes.flat, snaps_list, sigmas):
        im = ax.imshow(snaps[0, 0], origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax)
        ax.set_title(rf'$\sigma_b$ = {s}')
        ax.set_xticks([])
        ax.set_yticks([])
        ims.append(im)

    def update(i):
        i = min(i, n_frames - 1)
        for im, snaps in zip(ims, snaps_list):
            im.set_data(snaps[i, 0])
        fig.suptitle(f'Species 0 (sweep)   t = {t_labels[i]:.4f}')
        return ims

    anim = FuncAnimation(fig, update, frames=n_frames, interval=1000 / fps, blit=False)
    anim.save(path, writer='ffmpeg', fps=fps, dpi=dpi)
    plt.close(fig)
    print('saved', path)


In [ ]:
_n = snaps.shape[0]
_t_pp = np.empty(_n)
_t_pp[0] = 0.0
_t_pp[1:] = np.arange(1, _n) * SAVE_EVERY_PP * DT
movie_two_species(snaps, 'nr_pp.mp4', _t_pp, fps=10)


## 1b. Phase separation and predator-prey

Increase the off-diagonal `chi` so the mixture also has an equilibrium spinodal instability, then keep the same antisymmetric `B` forcing.

In [ ]:
# Strong chi -> spinodal instability; antisymmetric B (predator–prey) on top.
CHI_SCALE_PS = 5.0
chi_ps = CHI_SCALE_PS * jnp.array([[0.0, 1.0], [1.0, 0.0]])
b_ps = 120.0
B_ps = jnp.array([[0.0, +b_ps], [-b_ps, 0.0]])

N_STEPS_PS = 80_000
SAVE_EVERY_PS = 4_000
R_MU_PS = 150.0

key_ps = jax.random.PRNGKey(42)
_, k_ic_ps = jax.random.split(key_ps)
c0_ps = generate_initial_conditions(N_COM, N_GRID, beta=BETA, noise_strength=0.12, key=k_ic_ps)
x1d_ps = jnp.linspace(0.0, 1.0, N_GRID, endpoint=False)
strip_ps = 0.04 * jnp.sin(2.0 * jnp.pi * x1d_ps)
strip2d_ps = strip_ps[jnp.newaxis, :]
c0_ps = c0_ps.at[0].add(strip2d_ps).at[1].add(-strip2d_ps)

A_ps = float(jnp.abs(chi_ps).max()) * LMBDA
params_ps = build_params(
    chi_ps, N_COM, beta=BETA, lmbda=LMBDA, dt=DT, B=B_ps,
    omit_solvent_flux=True,
    A=A_ps,
    r_mu=R_MU_PS,
)

_, snaps_ps = simulate_with_snapshots(
    c0_ps, params_ps, N_GRID, n_steps=N_STEPS_PS, save_every=SAVE_EVERY_PS,
    activation='identity',
    include_initial=True,
)
print('snapshots (PS+PP):', snaps_ps.shape)
print('species 0 spatial std (first / last frame) =', float(snaps_ps[0, 0].std()), float(snaps_ps[-1, 0].std()))


In [ ]:
n_ps = snaps_ps.shape[0]
t_ps = np.empty(n_ps)
t_ps[0] = 0.0
t_ps[1:] = np.arange(1, n_ps) * SAVE_EVERY_PS * DT
k_ps = min(8, n_ps)
idx_ps = np.unique(np.rint(np.linspace(0, n_ps - 1, k_ps)).astype(int))
n_cols_ps = len(idx_ps)

fig, axes = plt.subplots(2, n_cols_ps, figsize=(2.2 * n_cols_ps, 4.5), squeeze=False)
vmins_ps = [float(snaps_ps[:, sp].min()) for sp in range(2)]
vmaxs_ps = [float(snaps_ps[:, sp].max()) for sp in range(2)]
for sp in range(2):
    if vmaxs_ps[sp] - vmins_ps[sp] < 1e-9:
        vmins_ps[sp] -= 0.02
        vmaxs_ps[sp] += 0.02
for col, i in enumerate(idx_ps):
    for sp in range(2):
        ax = axes[sp, col]
        ax.imshow(
            snaps_ps[i, sp], origin='lower',
            cmap=['Blues', 'Reds'][sp], vmin=vmins_ps[sp], vmax=vmaxs_ps[sp],
        )
        ax.set_xticks([])
        ax.set_yticks([])
        if sp == 0:
            ax.set_title(f't = {t_ps[i]:.3f}')
        if col == 0:
            ax.set_ylabel(f'species {sp}', rotation=0, labelpad=30)
fig.suptitle(
    rf'Spinodal $\chi$ (scale {CHI_SCALE_PS}) + antisymmetric $B$  ($b={b_ps:g}$)'
)
fig.tight_layout()
plt.show()


In [ ]:
movie_two_species(snaps_ps, 'nr_pp_plus_ps.mp4', t_ps, fps=10)


## 2. Random multi-species: nonreciprocal vs. equilibrium baseline

Fix `chi` (equilibrium thermodynamics) and the initial condition. Run two trajectories:

- `B = 0`  — relaxational dynamics, converges toward free-energy minima.
- `B != 0` random asymmetric — sustains non-equilibrium activity.


In [ ]:
N_COM = 10
N_GRID = 64
LMBDA = 0.01
DT = 5e-6
BETA = N_COM / (N_COM + 1.0)
N_STEPS = 120_000
SAVE_MULTI = 2_500

key = jax.random.PRNGKey(7)
k_chi, k_B, k_ic = jax.random.split(key, 3)

chi_N = generate_chi_matrix(N_COM, chi_mean=0.0, chi_std=3.5, key=k_chi)
B_N   = generate_B_matrix(N_COM, b_mean=0.0, b_std=8.0, key=k_B)

c0_N = generate_initial_conditions(N_COM, N_GRID, beta=BETA, key=k_ic)

A_multi = float(jnp.abs(chi_N).max()) * LMBDA
params_eq = build_params(chi_N, N_COM, beta=BETA, lmbda=LMBDA, dt=DT, A=A_multi)
params_nr = build_params(
    chi_N, N_COM, beta=BETA, lmbda=LMBDA, dt=DT, B=B_N, A=A_multi,
)

cf_eq, snaps_eq = simulate_with_snapshots(
    c0_N, params_eq, N_GRID, N_STEPS, save_every=SAVE_MULTI, include_initial=True,
)
cf_nr, snaps_nr = simulate_with_snapshots(
    c0_N, params_nr, N_GRID, N_STEPS, save_every=SAVE_MULTI,
    activation='identity', include_initial=True,
)
print('equilibrium   finite?', bool(jnp.all(jnp.isfinite(cf_eq))))
print('nonreciprocal finite?', bool(jnp.all(jnp.isfinite(cf_nr))))


In [ ]:
fig1, _ = plot_volume_fractions(cf_eq, vmin=0.0, vmax=0.4)
fig1.suptitle('Equilibrium (B = 0)')
fig2, _ = plot_volume_fractions(cf_nr, vmin=0.0, vmax=0.4)
fig2.suptitle('Nonreciprocal (B ≠ 0)')
plt.show()


In [ ]:
_nm = snaps_eq.shape[0]
_t_m = np.empty(_nm)
_t_m[0] = 0.0
_t_m[1:] = np.arange(1, _nm) * SAVE_MULTI * DT
movie_species0(snaps_eq, 'nr_multi_B0.mp4', _t_m, 'Equilibrium (B = 0); species 0', vmin=0.0, vmax=0.4)
movie_species0(snaps_nr, 'nr_multi_B.mp4', _t_m, 'Nonreciprocal (B ≠ 0); species 0', vmin=0.0, vmax=0.4)


## 3. Jacobian spectrum

Compare the symmetric Hessian `J` with the effective matrix `J + B`; asymmetric `B` produces complex eigenvalues.

In [ ]:
chi_s_zero = jnp.zeros((N_COM,))
r_ones = jnp.ones((N_COM,))
J = compute_jacobian(N_COM, BETA, chi_N, chi_s_zero, r_ones)

eig_eq = np.linalg.eigvals(np.asarray(J))
eig_nr = np.linalg.eigvals(np.asarray(J + B_N))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(eig_eq.real, eig_eq.imag, c='tab:blue', s=30)
axes[0].axvline(0, color='k', lw=0.6); axes[0].axhline(0, color='k', lw=0.6)
axes[0].set_title('Spectrum of J  (equilibrium)')
axes[0].set_xlabel('Re $\\lambda$'); axes[0].set_ylabel('Im $\\lambda$')

axes[1].scatter(eig_nr.real, eig_nr.imag, c='tab:red', s=30)
axes[1].axvline(0, color='k', lw=0.6); axes[1].axhline(0, color='k', lw=0.6)
axes[1].set_title('Spectrum of J + B  (nonreciprocal)')
axes[1].set_xlabel('Re $\\lambda$'); axes[1].set_ylabel('Im $\\lambda$')
fig.tight_layout()
plt.show()


## 4. Sweep of nonreciprocal strength

Fix `chi` and vary the scale of random `B` to compare equilibrium droplets with active, oscillatory patterns.

In [ ]:
sigmas = [0.0, 2.0, 4.0, 6.0, 8.0, 12.0]
finals = []
snaps_sigma_list = []
SAVE_SWEEP = 2_500
key_sweep = jax.random.PRNGKey(21)

for s in sigmas:
    B_s = generate_B_matrix(N_COM, 0.0, float(s), key_sweep)
    ps  = build_params(
        chi_N, N_COM, beta=BETA, lmbda=LMBDA, dt=DT, B=B_s, A=A_multi,
    )
    cf, snaps = simulate_with_snapshots(
        c0_N, ps, N_GRID, N_STEPS, save_every=SAVE_SWEEP,
        activation='identity', include_initial=True,
    )
    finals.append(np.asarray(cf))
    snaps_sigma_list.append(np.asarray(snaps))

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, s, cf in zip(axes.flat, sigmas, finals):
    # Show species 0 concentration as a diagnostic slice
    ax.imshow(cf[0], cmap='RdBu_r', origin='lower', vmin=0.0, vmax=0.3)
    ax.set_title(fr'$\sigma_b$ = {s}')
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Species 0 at t = N_STEPS*dt, sweeping nonreciprocity')
fig.tight_layout()
plt.show()


In [ ]:
_ns = snaps_sigma_list[0].shape[0]
_t_s = np.empty(_ns)
_t_s[0] = 0.0
_t_s[1:] = np.arange(1, _ns) * SAVE_SWEEP * DT
movie_sigma_grid(snaps_sigma_list, sigmas, 'nr_sigma_sweep.mp4', _t_s, fps=8, vmin=0.0, vmax=0.3)


## 5. Identity vs. tanh activation

Compare the same `B` with linear coupling and a saturating `tanh` activation.

In [ ]:
B_cmp = generate_B_matrix(N_COM, 0.0, 10.0, jax.random.PRNGKey(33))
params_cmp = build_params(
    chi_N, N_COM, beta=BETA, lmbda=LMBDA, dt=DT, B=B_cmp, A=A_multi,
)
SAVE_ACT = 2_500

cf_id, snaps_phi_id = simulate_with_snapshots(
    c0_N, params_cmp, N_GRID, N_STEPS, save_every=SAVE_ACT,
    activation='identity', include_initial=True,
)
cf_tanh, snaps_phi_tanh = simulate_with_snapshots(
    c0_N, params_cmp, N_GRID, N_STEPS, save_every=SAVE_ACT,
    activation='tanh', include_initial=True,
)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, cf, name in zip(axes, [cf_id, cf_tanh], ['identity', 'tanh']):
    ax.imshow(np.asarray(cf[0]), cmap='RdBu_r', origin='lower', vmin=0.0, vmax=0.3)
    ax.set_title(f"phi = {name}")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Same chi, B; different activation')
fig.tight_layout()
plt.show()


In [ ]:
_na = snaps_phi_id.shape[0]
_t_a = np.empty(_na)
_t_a[0] = 0.0
_t_a[1:] = np.arange(1, _na) * SAVE_ACT * DT
movie_species0(snaps_phi_id, 'nr_phi_identity.mp4', _t_a, 'phi = identity; species 0', vmin=0.0, vmax=0.3)
movie_species0(snaps_phi_tanh, 'nr_phi_tanh.mp4', _t_a, 'phi = tanh; species 0', vmin=0.0, vmax=0.3)
